## Manual analysis of total_sources 

List of sources that were selected to be part of our sample from a manual whitelist feature analysis and expansion to the whole raw sample 

### Set-up


In [1]:
import gc
import glob
import json
import os
from pathlib import Path

# Bridage des threads pour Numpy, Pandas et Scikit-Learn (laisse le CPU à DuckDB)
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"

import duckdb
from IPython.display import HTML, display
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns

# Configuration esthétique globale
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.family"] = "sans-serif"
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 30)

print("✔ Librairies importées et environnement configuré.")

✔ Librairies importées et environnement configuré.


In [2]:
# ==============================================================================
# 1. DÉFINITION DES CHEMINS (À adapter si tes fichiers sont dans un autre dossier)
# ==============================================================================
DATA_DIR = Path("/data/gdelt/gdelt_parquet_db")
SOURCE_MAP_PATH = Path("/data/gdelt/gdelt_sources_mapping.json")
RETAINED_IDS_PATH = Path("liste_ids_retenus.txt")
DOMAINS_PARQUET_PATH = "data/domains/domains_*.parquet"

# ==============================================================================
# 2. INITIALISATION ET PARAMÉTRAGE DE DUCKDB
# ==============================================================================
con = duckdb.connect()

# Réglage musclé pour ton grand serveur partagé (128 Go RAM / 16 threads en standard)
con.execute("PRAGMA memory_limit='128GB'")
con.execute("PRAGMA threads=16")

# 3. Chargement du dictionnaire JSON (Traduction ID <-> Nom de domaine)
with open(SOURCE_MAP_PATH, "r", encoding="utf-8") as f:
  source_map = json.load(f)

src_df = pd.DataFrame({
    "SourceCommonName_ID": [int(k) for k in source_map["id_to_source"].keys()],
    "SourceCommonName": list(source_map["id_to_source"].values()),
})
con.register("src_map", src_df)

# 4. Chargement de TA LISTE PROPRE d'ID retenus dans une table DuckDB dédiée
con.execute(f"""
    CREATE OR REPLACE TABLE retained_ids AS 
    SELECT column0::BIGINT AS id 
    FROM read_csv('{RETAINED_IDS_PATH}', header=False)
""")

nb_ids = con.execute("SELECT COUNT(*) FROM retained_ids").fetchone()[0]
print(
    f"✔ DuckDB initialisé. Table 'retained_ids' chargée ({nb_ids:,} médias"
    " légitimes)."
)

✔ DuckDB initialisé. Table 'retained_ids' chargée (13,334 médias légitimes).


In [3]:
glob_pattern = str(DATA_DIR / "gdelt_*.parquet")

print(
    "⏳ Création de la vue maîtresse 'gkg_clean' (Filtrage sur échantillon"
    " propre & enrichissement)..."
)

con.execute(f"""
    CREATE OR REPLACE VIEW gkg_clean AS
    
    WITH raw_filtered AS (
        -- 1. Nettoyage initial : dates valides, exclusion ligne corrompue et thèmes vides
        SELECT *
        FROM read_parquet('{glob_pattern}')
        WHERE regexp_matches(CAST(DATE AS VARCHAR), '^\d{{14}}$')
          AND GKGRECORDID != '20210925181500-T1111'
          AND EnhancedThemes IS NOT NULL 
          AND EnhancedThemes != ''
    ),
    
    repaired AS (
        -- 2. Réparation de l'identifiant (si 0 ou NULL, on tente de matcher le nom de domaine via src_map)
        SELECT 
            r.* EXCLUDE (SourceCommonName_ID),
            CASE 
                WHEN COALESCE(r.SourceCommonName_ID, 0) = 0 THEN m.SourceCommonName_ID
                ELSE r.SourceCommonName_ID
            END AS SourceCommonName_ID
        FROM raw_filtered r
        LEFT JOIN src_map m 
          ON RTRIM(regexp_extract(r.DocumentIdentifier, 'https?://(?:www\.)?([^/?:]+)', 1), '.') = m.SourceCommonName
    )
    
    -- 3. FILTRAGE STRICT SUR L'ÉCHANTILLON ET ENRICHISSEMENT WIKIDATA
    SELECT 
        rep.*,
        w.medialabel,
        w.typelabel,
        w.countrylabel,
        w.inception,
        CASE WHEN w.Src_ID IS NOT NULL THEN 1 ELSE 0 END AS is_wiki
    FROM repaired rep
    -- 🔥 LE FILTRE CLÉ : Jointure interne avec ta liste des sources retenues !
    INNER JOIN retained_ids rid 
      ON rep.SourceCommonName_ID = rid.id
    -- Enrichissement optionnel : si la source est dans Wikidata, on récupère ses labels
    LEFT JOIN (
        SELECT 
            id AS Src_ID, 
            medialabel, 
            typelabel, 
            countrylabel, 
            inception
        FROM read_parquet('{DOMAINS_PARQUET_PATH}')
        QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY inception ASC, countrylabel ASC) = 1
    ) w ON rep.SourceCommonName_ID = w.Src_ID;
""")

print(
    "✔ Vue 'gkg_clean' prête ! Elle ne contient que les articles de ton"
    " échantillon sélectionné."
)

⏳ Création de la vue maîtresse 'gkg_clean' (Filtrage sur échantillon propre & enrichissement)...
✔ Vue 'gkg_clean' prête ! Elle ne contient que les articles de ton échantillon sélectionné.


<>:57: SyntaxWarning: invalid escape sequence '\d'
<>:57: SyntaxWarning: invalid escape sequence '\.'
<>:57: SyntaxWarning: invalid escape sequence '\d'
<>:57: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_327808/1892980978.py:57: SyntaxWarning: invalid escape sequence '\d'
  """)
/tmp/ipykernel_327808/1892980978.py:57: SyntaxWarning: invalid escape sequence '\.'
  """)


In [4]:
print("⏳ Calcul de la volumétrie globale sur ton échantillon...")

# 1. Requête rapide de vérification
df_check = con.execute("""
    SELECT 
        COUNT(*) AS total_articles_disponibles,
        COUNT(DISTINCT SourceCommonName_ID) AS medias_actifs,
        MIN(strptime(substr(CAST(DATE AS VARCHAR), 1, 8), '%Y%m%d')::DATE) AS date_min,
        MAX(strptime(substr(CAST(DATE AS VARCHAR), 1, 8), '%Y%m%d')::DATE) AS date_max,
        SUM(is_wiki) AS articles_avec_metadata_wiki
    FROM gkg_clean;
""").df()

# 2. Affichage du résumé
display(df_check.style.format({
    "total_articles_disponibles": "{:,.0f}",
    "medias_actifs": "{:,.0f}",
    "articles_avec_metadata_wiki": "{:,.0f}"
}))

# 3. Aperçu des 3 premiers articles au hasard pour vérifier la structure
print("\n👀 Aperçu de 3 articles au hasard dans la vue propre :")
display(con.execute("""
    SELECT 
        GKGRECORDID, 
        SourceCommonName_ID, 
        medialabel, 
        countrylabel, 
        substr(DocumentIdentifier, 1, 60) || '...' AS URL,
        substr(EnhancedThemes, 1, 80) || '...' AS Themes_extrait
    FROM gkg_clean 
    USING SAMPLE 3;
""").df())

⏳ Calcul de la volumétrie globale sur ton échantillon...


,total_articles_disponibles,medias_actifs,date_min,date_max,articles_avec_metadata_wiki
0,"1,136,023,125","13,334",2015-02-18 00:00:00,2026-06-19 00:00:00,"459,677,212"



👀 Aperçu de 3 articles au hasard dans la vue propre :


,GKGRECORDID,SourceCommonName_ID,medialabel,countrylabel,URL,Themes_extrait
0,20150303134500-438,5349,NaN,NaN,http://www.beijingbulletin.com/index.php/sid/2...,"TAX_FNCACT_LEADER,145;TAX_FNCACT_LEADER,935;..."
1,20150219020000-1038,22,NaN,NaN,http://www.heraldstandard.com/entertainment_ap...,"TAX_FNCACT,19;TAX_FNCACT,651;TAX_FNCACT,2593;..."
2,20150219020000-1928,694,Western Telegraph,,http://www.westerntelegraph.co.uk/news/nationa...,"KIDNAP,2475;MANMADE_DISASTER_IMPLIED,1688;TAX_..."


### Random sample analysis 

#### Analysis of 100 sources, randomly chosen among all the selected sources 

In [6]:
print("Tirage aléatoire de 1000 sources dans l'échantillon retenu...\n")

# 1. Requête SQL : Jointure avec le dictionnaire des noms et tirage aléatoire
df_sample_100 = con.execute("""
    SELECT 
        r.id AS SourceCommonName_ID,
        COALESCE(m.SourceCommonName, 'Domaine inconnu (' || r.id || ')') AS SourceCommonName
    FROM retained_ids r
    LEFT JOIN src_map m ON r.id = m.SourceCommonName_ID
    ORDER BY random()
    LIMIT 100;
""").df()

# 2. Réinitialisation de l'index pour avoir un comptage propre de 1 à 100
df_sample_100.index = range(1, len(df_sample_100) + 1)
df_sample_100.index.name = '#'

# 3. AFFICHAGE EN GRILLE DE TEXTE (4 colonnes) pour un balayage visuel ultra-rapide
domaines = df_sample_100['SourceCommonName'].tolist()
n_cols = 4
n_rows = (len(domaines) + n_cols - 1) // n_cols

print("BALAYAGE RAPIDE — 1000 médias tirés au hasard :\n" + "="*85)
for r in range(n_rows):
    row_items = []
    for c in range(n_cols):
        idx = r + c * n_rows
        if idx < len(domaines):
            # On formate chaque nom sur 20 caractères pour aligner les colonnes
            row_items.append(f"{idx+1:3d}. {domaines[idx]:<18}")
    print(" | ".join(row_items))

print("\n" + "="*85)

# 4. AFFICHAGE DU DATAFRAME COMPLET (Pour inspection détaillée si besoin)
# On force Pandas à afficher les 100 lignes sans tronquer
with pd.option_context('display.max_rows', 1000):
    display(df_sample_100.style.set_properties(**{
        'font-weight': 'bold', 
        'text-align': 'left'
    }))

Tirage aléatoire de 1000 sources dans l'échantillon retenu...

BALAYAGE RAPIDE — 1000 médias tirés au hasard :
  1. wapa.pe            |  26. turkiyegazetesi.com.tr |  51. teleradioerre.it   |  76. fudzilla.com      
  2. ecoportal.net      |  27. wcjc.com           |  52. lavozdemichoacan.com.mx |  77. balticbusinessnews.com
  3. fool.ca            |  28. nikvesti.com       |  53. istanbulhaber.com.tr |  78. wcti12.com        
  4. epochtimes.com.tw  |  29. oskaloosa.com      |  54. diarioeyipantla.com |  79. sciencerecorder.com
  5. politis.fr         |  30. zive.sk            |  55. nogalesinternational.com |  80. rttnews.com       
  6. hl.co.uk           |  31. clutchfans.net     |  56. jpnn.com           |  81. economynext.com   
  7. bclocalnews.com    |  32. sonapresse.com     |  57. mtexpress.com      |  82. arka.am           
  8. pnp.de             |  33. turmush.kg         |  58. ukrinform.ua       |  83. observer-reporter.com
  9. goldentranscript.net |  34. khm.depo.ua   

,SourceCommonName_ID,SourceCommonName
#,,
1,28616,wapa.pe
2,29714,ecoportal.net
3,8909,fool.ca
4,61100,epochtimes.com.tw
5,28365,politis.fr
6,20541,hl.co.uk
7,159809,bclocalnews.com
8,26153,pnp.de
9,125597,goldentranscript.net


"You are a research assistant in economics and I give you this list of 1000 sources randmoly chosen from my sample. I want to know how many are not newspapers that discuss economic, politic and business topics"


### Descriptive stats 

In [4]:
audit_query = """
-- 1. VOLUME ET UNICITÉ
SELECT '1. Volume & Unicité' AS Categorie, 'Nombre total d''articles' AS Indicateur, CAST(COUNT(*) AS VARCHAR) AS Valeur, 'ℹ️' AS Statut FROM gkg_clean
UNION ALL
SELECT '1. Volume & Unicité', 'Doublons sur GKGRECORDID', CAST(COUNT(*) - COUNT(DISTINCT GKGRECORDID) AS VARCHAR), CASE WHEN COUNT(*) = COUNT(DISTINCT GKGRECORDID) THEN '✅' ELSE '⚠️' END FROM gkg_clean

-- 2. CONFORMITÉ DES FORMATS
UNION ALL
SELECT '2. Conformité', 'URLs invalides (pas HTTP) sur Source=1', CAST(COUNT(*) AS VARCHAR), CASE WHEN COUNT(*) = 0 THEN '✅' ELSE '⚠️' END FROM gkg_clean WHERE SourceCollectionIdentifier = 1 AND DocumentIdentifier NOT ILIKE 'http%'
UNION ALL
SELECT '2. Conformité', 'IsTranslingual (valeurs hors 0/1)', CAST(COUNT(*) AS VARCHAR), CASE WHEN COUNT(*) = 0 THEN '✅' ELSE '⚠️' END FROM gkg_clean WHERE IsTranslingual NOT IN (0, 1)

-- 3. COMPLÉTUDE
UNION ALL
SELECT '3. Complétude (Critique)', 'GKGRECORDID / DATE / URL (% Vides)', CAST(ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM gkg_clean), 4) AS VARCHAR) || ' %', CASE WHEN COUNT(*) = 0 THEN '✅' ELSE '⚠️' END FROM gkg_clean WHERE GKGRECORDID IS NULL OR DATE IS NULL OR DocumentIdentifier IS NULL OR DocumentIdentifier = ''
UNION ALL
SELECT '3. Complétude (Information)', 'EnhancedThemes (% Vides)', CAST(ROUND(100.0 * SUM(CASE WHEN EnhancedThemes IS NULL OR EnhancedThemes = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS VARCHAR) || ' %', 'ℹ️' FROM gkg_clean
UNION ALL
SELECT '3. Complétude (Information)', 'EnhancedLocations (% Vides)', CAST(ROUND(100.0 * SUM(CASE WHEN EnhancedLocations IS NULL OR EnhancedLocations = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS VARCHAR) || ' %', 'ℹ️' FROM gkg_clean
UNION ALL
SELECT '3. Complétude (Information)', 'Persons (% Vides)', CAST(ROUND(100.0 * SUM(CASE WHEN Persons IS NULL OR Persons = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS VARCHAR) || ' %', 'ℹ️' FROM gkg_clean
UNION ALL
SELECT '3. Complétude (Information)', 'Organizations (% Vides)', CAST(ROUND(100.0 * SUM(CASE WHEN Organizations IS NULL OR Organizations = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS VARCHAR) || ' %', 'ℹ️' FROM gkg_clean

-- 4. PROFILING NUMÉRIQUE
UNION ALL
SELECT '4. Profiling Numérique', 'Tone (Min | Q1 | Médiane | Moyenne | Q3 | Max)', 
    CAST(ROUND(MIN(Tone), 1) AS VARCHAR) || ' | ' || CAST(ROUND(APPROX_QUANTILE(Tone, 0.25), 1) AS VARCHAR) || ' | ' || CAST(ROUND(APPROX_QUANTILE(Tone, 0.50), 1) AS VARCHAR) || ' | ' || CAST(ROUND(AVG(Tone), 2) AS VARCHAR) || ' | ' || CAST(ROUND(APPROX_QUANTILE(Tone, 0.75), 1) AS VARCHAR) || ' | ' || CAST(ROUND(MAX(Tone), 1) AS VARCHAR), 
    CASE WHEN MIN(Tone) >= -100 AND MAX(Tone) <= 100 THEN '✅' ELSE '⚠️' END FROM gkg_clean
UNION ALL
SELECT '4. Profiling Numérique', 'WordCount (Min | Q1 | Médiane | Moyenne | Q3 | Max)', 
    CAST(MIN(WordCount) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(WordCount, 0.25) AS INTEGER) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(WordCount, 0.50) AS INTEGER) AS VARCHAR) || ' | ' || CAST(ROUND(AVG(WordCount), 0) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(WordCount, 0.75) AS INTEGER) AS VARCHAR) || ' | ' || CAST(MAX(WordCount) AS VARCHAR), 
    CASE WHEN MIN(WordCount) >= 0 THEN '✅' ELSE '⚠️' END FROM gkg_clean


-- 5. PROFILING SÉMANTIQUE (Entités)
UNION ALL
SELECT '5. Profiling Sémantique', 'Thèmes uniques (Min | Q1 | Médiane | Moyenne | Q3 | Max)', 
    CAST(MIN(val) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(val, 0.25) AS INTEGER) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(val, 0.50) AS INTEGER) AS VARCHAR) || ' | ' || CAST(ROUND(AVG(val), 1) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(val, 0.75) AS INTEGER) AS VARCHAR) || ' | ' || CAST(MAX(val) AS VARCHAR), 
    'ℹ️' FROM (SELECT CASE WHEN EnhancedThemes = '' THEN 0 ELSE ARRAY_LENGTH(string_split(EnhancedThemes, ';')) END AS val FROM gkg_clean)
UNION ALL
SELECT '5. Profiling Sémantique', 'Personnes uniques (Min | Q1 | Médiane | Moyenne | Q3 | Max)', 
    CAST(MIN(val) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(val, 0.25) AS INTEGER) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(val, 0.50) AS INTEGER) AS VARCHAR) || ' | ' || CAST(ROUND(AVG(val), 1) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(val, 0.75) AS INTEGER) AS VARCHAR) || ' | ' || CAST(MAX(val) AS VARCHAR), 
    'ℹ️' FROM (SELECT CASE WHEN Persons = '' THEN 0 ELSE ARRAY_LENGTH(string_split(Persons, ';')) END AS val FROM gkg_clean)
UNION ALL
SELECT '5. Profiling Sémantique', 'Organisations uniques (Min | Q1 | Médiane | Moyenne | Q3 | Max)', 
    CAST(MIN(val) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(val, 0.25) AS INTEGER) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(val, 0.50) AS INTEGER) AS VARCHAR) || ' | ' || CAST(ROUND(AVG(val), 1) AS VARCHAR) || ' | ' || CAST(CAST(APPROX_QUANTILE(val, 0.75) AS INTEGER) AS VARCHAR) || ' | ' || CAST(MAX(val) AS VARCHAR), 
    'ℹ️' FROM (SELECT CASE WHEN Organizations = '' THEN 0 ELSE ARRAY_LENGTH(string_split(Organizations, ';')) END AS val FROM gkg_clean)

ORDER BY Categorie, Indicateur;
"""

dashboard_sante = con.execute(audit_query).df()

styled_dashboard = dashboard_sante.style.set_properties(**{'text-align': 'left'}, subset=['Categorie', 'Indicateur', 'Valeur'])\
                                        .set_properties(**{'text-align': 'center'}, subset=['Statut'])\
                                        .hide(axis="index")
display(styled_dashboard)

Categorie,Indicateur,Valeur,Statut
1. Volume & Unicité,Doublons sur GKGRECORDID,0,✅
1. Volume & Unicité,Nombre total d'articles,1136023125,ℹ️
2. Conformité,IsTranslingual (valeurs hors 0/1),0,✅
2. Conformité,URLs invalides (pas HTTP) sur Source=1,0,✅
3. Complétude (Critique),GKGRECORDID / DATE / URL (% Vides),0.0 %,✅
3. Complétude (Information),EnhancedLocations (% Vides),19.93 %,ℹ️
3. Complétude (Information),EnhancedThemes (% Vides),0.0 %,ℹ️
3. Complétude (Information),Organizations (% Vides),31.78 %,ℹ️
3. Complétude (Information),Persons (% Vides),46.14 %,ℹ️
4. Profiling Numérique,Tone (Min | Q1 | Médiane | Moyenne | Q3 | Max),-100.0 | -3.6 | -0.9 | -1.19 | 1.3 | 100.0,✅


In [7]:
import pandas as pd

print("🔍 INVESTIGATION DES GIGA-OUTLIERS (URLs COMPLÈTES)")
print("="*85)

# On demande temporairement à Pandas de ne pas tronquer le contenu des colonnes
with pd.option_context('display.max_colwidth', None):

    # 1. Les articles minuscules (Ex: WordCount = 1)
    print("\n🤏 OUTLIERS BAS : WordCount très faible (<= 10 mots)")
    display(con.execute("""
        SELECT GKGRECORDID, WordCount, DocumentIdentifier AS URL
        FROM gkg_clean 
        WHERE WordCount <= 10 
        ORDER BY WordCount ASC 
        LIMIT 5;
    """).df())

    # 2. Les articles gigantesques (Ex: WordCount > 50 000)
    print("\n🏔️ OUTLIERS HAUTS : WordCount gigantesque (> 50 000 mots)")
    display(con.execute("""
        SELECT GKGRECORDID, WordCount, DocumentIdentifier AS URL
        FROM gkg_clean 
        WHERE WordCount > 5000
        ORDER BY WordCount DESC 
        LIMIT 5;
    """).df())

    # 3. Les surcharges d'entités (Organisations / Personnes / Thèmes)
    print("\n🐙 OUTLIERS HAUTS : Surcharge d'Organisations (> 500)")
    display(con.execute("""
        SELECT 
            GKGRECORDID, 
            ARRAY_LENGTH(string_split(Organizations, ';')) AS Nb_Orgs,
            DocumentIdentifier AS URL
        FROM gkg_clean 
        WHERE Organizations != '' AND ARRAY_LENGTH(string_split(Organizations, ';')) > 100
        ORDER BY Nb_Orgs DESC 
        LIMIT 5;
    """).df())

    print("\n🧠 OUTLIERS HAUTS : Surcharge de Thèmes (> 10 000)")
    display(con.execute("""
        SELECT 
            GKGRECORDID, 
            ARRAY_LENGTH(string_split(EnhancedThemes, ';')) AS Nb_Themes,
            DocumentIdentifier AS URL
        FROM gkg_clean 
        WHERE EnhancedThemes != '' AND ARRAY_LENGTH(string_split(EnhancedThemes, ';')) > 1000
        ORDER BY Nb_Themes DESC 
        LIMIT 5;
    """).df())

🔍 INVESTIGATION DES GIGA-OUTLIERS (URLs COMPLÈTES)

🤏 OUTLIERS BAS : WordCount très faible (<= 10 mots)


,GKGRECORDID,WordCount,URL
0,20160602100000-T1650,1,http://ekabu.ru/fun/123100-ulybaytes-gospoda-ulybaytes-30-foto.html
1,20161018080000-T1369,1,http://ekabu.ru/fun/130513-ulybaytes-gospoda-ulybaytes-30-foto.html
2,20160222004500-T1827,1,http://www.pluska.sk/regiony/vychodne-slovensko/preco-idem-volit-danica-caisova-sexuologicka-psychiatricka.html
3,20150219201500-T2530,1,http://www.h-avis.no/Heder_til_Aronsen-5-62-17623.html
4,20150414013000-1684,1,http://www.opednews.com/populum/linkrss.php?f=Deputy-who-shot-and-killed-in-Daily-Kos-150413-573.html



🏔️ OUTLIERS HAUTS : WordCount gigantesque (> 50 000 mots)


,GKGRECORDID,WordCount,URL
0,20150410023000-263,747966,http://www.sloveniatimes.com/palm-sunday-celebrated-in-slovenia
1,20241028134500-T2061,707869,https://blog.wenxuecity.com/myblog/80439/202410/24877.html
2,20240906070000-336,526987,https://www.federalregister.gov/documents/2024/07/31/2024-14828/medicare-and-medicaid-programs-cy-2025-payment-policies-under-the-physician-fee-schedule-and-other
3,20150528131500-1910,469593,http://www.thetelegraphandargus.co.uk/news/broadway/11877909.display/
4,20171117181500-1820,433723,https://www.federalregister.gov/documents/2017/11/17/2017-21808/payday-vehicle-title-and-certain-high-cost-installment-loans



🐙 OUTLIERS HAUTS : Surcharge d'Organisations (> 500)


RuntimeError: Query interrupted

In [9]:
import pandas as pd
from IPython.display import display

print("⏳ Calcul de la distribution très fine (analyse des extrêmes)...")

query_quantiles = """
WITH metrics AS (
    SELECT 
        WordCount,
        CASE WHEN Organizations = '' OR Organizations IS NULL THEN 0 ELSE ARRAY_LENGTH(string_split(Organizations, ';')) END AS OrgCount,
        CASE WHEN Persons = '' OR Persons IS NULL THEN 0 ELSE ARRAY_LENGTH(string_split(Persons, ';')) END AS PersonCount,
        CASE WHEN EnhancedThemes = '' OR EnhancedThemes IS NULL THEN 0 ELSE ARRAY_LENGTH(string_split(EnhancedThemes, ';')) END AS ThemeCount,
        CASE WHEN EnhancedLocations = '' OR EnhancedLocations IS NULL THEN 0 ELSE ARRAY_LENGTH(string_split(EnhancedLocations, ';')) END AS LocCount
    FROM gkg_clean
)
SELECT 
    -- Ajout des quantiles bas (0.0, 0.0001, 0.001, 0.005) pour voir le comportement près de zéro
    APPROX_QUANTILE(WordCount,   [0.0, 0.0001, 0.001, 0.005, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999, 0.9999, 1.0]) AS wc_q,
    APPROX_QUANTILE(OrgCount,    [0.0, 0.0001, 0.001, 0.005, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999, 0.9999, 1.0]) AS org_q,
    APPROX_QUANTILE(PersonCount, [0.0, 0.0001, 0.001, 0.005, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999, 0.9999, 1.0]) AS pers_q,
    APPROX_QUANTILE(ThemeCount,  [0.0, 0.0001, 0.001, 0.005, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999, 0.9999, 1.0]) AS theme_q,
    APPROX_QUANTILE(LocCount,    [0.0, 0.0001, 0.001, 0.005, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999, 0.9999, 1.0]) AS loc_q
FROM metrics;
"""

# Exécution de la requête
df_quantiles = con.execute(query_quantiles).df()

# Définition des labels pour nos pourcentages, adaptés à la nouvelle requête
percentiles = [
    'Min (0%)', '0.01%', '0.1%', '0.5%', '1%', '5%', '10%', '25%', 
    '50% (Med)', '75%', '90%', '95%', '99%', '99.5%', '99.9%', '99.99%', 'Max'
]

# Création d'un DataFrame propre en transposant les résultats
df_fine_distribution = pd.DataFrame({
    'Mots (WordCount)': df_quantiles['wc_q'][0],
    'Localisations uniques': df_quantiles['loc_q'][0],
    'Organisations uniques': df_quantiles['org_q'][0],
    'Personnes uniques': df_quantiles['pers_q'][0],
    'Thèmes uniques': df_quantiles['theme_q'][0]
}, index=percentiles).T

# Affichage avec un dégradé de couleurs (heatmap) par ligne
display(
    df_fine_distribution.style.format("{:,.0f}")
    .background_gradient(cmap='YlOrRd', axis=1)
    .set_caption("📊 Distribution granulaire des métadonnées (repérage des Outliers Hauts et Bas)")
)

⏳ Calcul de la distribution très fine (analyse des extrêmes)...


,Min (0%),0.01%,0.1%,0.5%,1%,5%,10%,25%,50% (Med),75%,90%,95%,99%,99.5%,99.9%,99.99%,Max
Mots (WordCount),6,8,16,28,39,77,100,165,295,501,794,"1,048","1,997","2,658","5,444","15,262","39,841"
Localisations uniques,0,0,0,0,0,0,0,1,4,10,21,31,67,89,175,935,935
Organisations uniques,0,0,0,0,0,0,0,0,1,3,5,7,12,15,28,117,117
Personnes uniques,0,0,0,0,0,0,0,0,1,2,5,7,14,18,35,82,125
Thèmes uniques,2,2,2,2,2,4,7,16,31,57,94,126,225,285,517,"1,156","1,904"
